## Corrected fine-tuned Qwen3 baseline for RQ1 significance testing — DBpedia

Fixes a bug in `scripts/eval_rq1_export.py --model-type sentence-transformer`:
loading the LoRA adapter directory directly with `SentenceTransformer(path)`
silently falls back to **mean pooling** (a bare adapter directory has no
`modules.json`, so sentence-transformers' `_load_auto_model` fallback kicks
in), instead of the **last-token pooling** this model was actually trained
and evaluated with everywhere else
(`notebooks/finetuning/0{4,5}_qwen3_contrastive_finetune_square*.ipynb`,
`notebooks/caching/caching_simulation_dbpedia.ipynb`). No error is raised —
it silently produces different, worse embeddings.

This notebook redoes the fine-tuned-Qwen3 baseline correctly (base model +
PEFT adapter + last-token pooling, matching the caching notebook) and reruns
the full significance pipeline end-to-end, entirely in-notebook (no
`python scripts/*.py` calls needed):

1. Correctly re-embed the RQ1 DBpedia test corpus with the fine-tuned
   Qwen3 adapter, export per-anchor MRR/nDCG — overwrites
   `results/significance/dbpedia_baseline_qwen3ft.json`.
2. Load the 5 existing SparqlCE seed exports
   (`dbpedia_seed{42..46}_full.json` — unaffected by the bug, since those were
   always loaded via `SPARQLEmbedder`, not `SentenceTransformer`; no need to
   rerun training).
3. Recompute cross-seed stats + the paired Wilcoxon significance test against
   the corrected baseline — overwrites
   `results/significance/dbpedia_full_summary.json`.

Requires `peft`, `transformers`, `scipy` (same environment as the caching
notebook) and the adapter at
`scripts/qwen_ft/qwen3_square_dbpedia/best_adapter`.


In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import torch

# ---- Project root (works from notebooks/evaluation-RQ1/DBpedia/ or anywhere in the repo) ----
_PROJECT_ROOT = Path.cwd()
while not (_PROJECT_ROOT / "datasets").exists():
    assert _PROJECT_ROOT != _PROJECT_ROOT.parent, "could not locate repo root"
    _PROJECT_ROOT = _PROJECT_ROOT.parent

for p in (
    _PROJECT_ROOT,
    _PROJECT_ROOT / "notebooks" / "evaluation-RQ1",
    _PROJECT_ROOT / "scripts",
):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(
    f"project root: {_PROJECT_ROOT}\n"
    f"device: {DEVICE}"
)


In [ ]:
# ---- Config ----
KG = "dbpedia"

TESTSET_PATH = (
    _PROJECT_ROOT
    / "datasets"
    / "dbpedia_testset_RQ1.json"
)

ADAPTER_PATH = (
    _PROJECT_ROOT
    / "scripts"
    / "qwen_ft"
    / "qwen3_square_dbpedia"
    / "best_adapter"
)

BASELINE_OUT = (
    _PROJECT_ROOT
    / "results"
    / "significance"
    / "dbpedia_baseline_qwen3ft.json"
)

SEED_GLOB = str(
    _PROJECT_ROOT
    / "results"
    / "significance"
    / "dbpedia_seed*_full.json"
)

SUMMARY_OUT = (
    _PROJECT_ROOT
    / "results"
    / "significance"
    / "dbpedia_full_summary.json"
)

if not ADAPTER_PATH.exists():
    raise FileNotFoundError(
        f"Adapter not found: {ADAPTER_PATH}\n"
        "Verify qwen3_square_dbpedia/best_adapter was produced by the "
        "fine-tuning notebook and is at this path."
    )

if not (ADAPTER_PATH / "adapter_config.json").exists():
    raise FileNotFoundError(
        f"Missing adapter_config.json in {ADAPTER_PATH}"
    )

print(f"KG: {KG}")
print(f"adapter: {ADAPTER_PATH}")
print(f"baseline output: {BASELINE_OUT}")
print(f"summary output: {SUMMARY_OUT}")


In [ ]:
# ============================================================================
# 1. Load the RQ1 test corpus (same data-loading helpers eval_rq1_export.py
#    and evaluate_rq1.py use -- only the fine-tuned-Qwen3 embedding step was
#    buggy, this part is unaffected).
# ============================================================================
from evaluate_rq1 import (
    load_triplets,
    build_global_corpus,
    build_eval_items,
    build_binary_relevance_maps,
    build_sorted_pools_global,
)
from eval_rq1_export import (
    build_emb_map_peft_qwen3,
    per_anchor_mrr,
    per_anchor_ndcg,
    TOPKS,
)

raw = load_triplets(str(TESTSET_PATH))
global_corpus, seen_q = build_global_corpus(raw)
eval_items = build_eval_items(raw, global_corpus, seen_q)
rel_maps = build_binary_relevance_maps(eval_items, global_corpus)

print(
    f"corpus: {len(global_corpus)} queries, "
    f"{len(eval_items)} eval anchors"
)


In [ ]:
# ============================================================================
# 2. Correctly embed the RQ1 corpus with the fine-tuned Qwen3 adapter
#
# build_emb_map_peft_qwen3 (scripts/eval_rq1_export.py) reconstructs the
# exact training-time architecture: base model via AutoModel, LoRA adapter
# via PeftModel.from_pretrained, last-token pooling, L2 normalization --
# the same approach already used correctly in the caching notebook. This is
# what SentenceTransformer(adapter_path) silently failed to do (see intro).
# ============================================================================
emb_map = build_emb_map_peft_qwen3(
    ADAPTER_PATH,
    global_corpus,
    batch_size=8,
)

sorted_pools = build_sorted_pools_global(
    emb_map, eval_items, global_corpus
)

anchors = [a for a, _, _ in eval_items]
per_anchor = {}
aggregate = {}

for k in TOPKS:
    mrr_list = per_anchor_mrr(sorted_pools, eval_items, k)
    ndcg_list = per_anchor_ndcg(sorted_pools, rel_maps, k)
    per_anchor[f"mrr@{k}"] = mrr_list
    per_anchor[f"ndcg@{k}"] = ndcg_list
    aggregate[f"mrr@{k}"] = float(np.mean(mrr_list))
    aggregate[f"ndcg@{k}"] = float(np.mean(ndcg_list))

baseline_run = {
    "kg": KG,
    "model_type": "peft-qwen3",
    "model_id": str(ADAPTER_PATH),
    "seed": None,
    "ablation": "baseline-qwen3ft",
    "anchors": anchors,
    "aggregate": aggregate,
    "per_anchor": per_anchor,
}

BASELINE_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(BASELINE_OUT, "w") as f:
    json.dump(baseline_run, f, indent=2)

print(f"Saved corrected baseline to {BASELINE_OUT}\n")
print("Aggregate (corrected, last-token pooling):")
for metric, val in aggregate.items():
    print(f"  {metric:10s} {val:.4f}")


In [ ]:
# ============================================================================
# 3. Recompute cross-seed stats + significance vs. the corrected baseline
#
# Reuses compute_significance.py's functions directly (those were always
# correct -- only the baseline *embedding* step had the pooling bug, not the
# statistics). The 5 SparqlCE seed files are untouched by this fix.
# ============================================================================
from compute_significance import (
    load_runs,
    cross_seed_stats,
    representative_run,
    paired_test,
)

sparqlce_runs = load_runs(SEED_GLOB)
cross_seed = cross_seed_stats(sparqlce_runs)
rep_run = representative_run(sparqlce_runs)
sig = paired_test(rep_run, baseline_run)

summary = {
    "n_seeds": len(sparqlce_runs),
    "seeds": [r.get("seed") for r in sparqlce_runs],
    "cross_seed_stats": cross_seed,
    "representative_seed": rep_run.get("seed"),
    "baseline_model": baseline_run.get("model_id"),
    "significance_vs_baseline": sig,
}

with open(SUMMARY_OUT, "w") as f:
    json.dump(summary, f, indent=2)

print(f"n_seeds={summary['n_seeds']}  seeds={summary['seeds']}")
print("\nCross-seed mean +/- std:")
for metric, s in cross_seed.items():
    print(f"  {metric:10s} {s['mean']:.4f} +/- {s['std']:.4f}  (n={s['n_seeds']})")

print(
    f"\nSignificance vs. {summary['baseline_model']} "
    f"(seed={rep_run.get('seed')}, Wilcoxon signed-rank, one-sided):"
)
for metric, s in sig.items():
    p = s.get("p_value")
    flag = "*" if s.get("significant_at_0.05") else " "
    print(f"  {metric:10s} p={p}  mean_diff={s.get('mean_diff'):.4f} {flag}")

print(f"\nSaved to {SUMMARY_OUT}")


### Notes

- This overwrites `dbpedia_baseline_qwen3ft.json` and `dbpedia_full_summary.json`
  in place. `git diff results/significance/` after running will show exactly
  how the corrected numbers differ from the buggy (mean-pooled) ones —
  worth checking before committing.
- Expect `mrr@10` for the corrected baseline to land near **0.1091**
  (the Qwen3-Emb-0.6B (fine-tuned) row in `tab:dbpedia_ranking_new`), not
  the old buggy ~0.098 — that mismatch was the tell something was wrong.
- After running both the DBpedia and Wikidata versions of this notebook,
  update `results/significance/README.md` and the "Significance and seed
  variance" paragraph in
  `sparql-embeddings-latex/Sections/experiments-v2.tex` with the corrected
  numbers.
